In [173]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.power import TTestIndPower

In [174]:
df = pd.concat(pd.read_excel('data/raw/online_retail_II.xlsx',sheet_name=None), ignore_index=True)

In [175]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [176]:
df.shape

(1067371, 8)

In [177]:
df = df.rename(columns = {"Invoice":"invoice","StockCode":"stock_code","Description":"description","Quantity":"quantity","InvoiceDate":"invoice_date","Price":"price","Customer ID":"customer_id","Country":"country"})

In [178]:
df.dtypes

invoice                 object
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[us]
price                  float64
customer_id            float64
country                    str
dtype: object

In [179]:
df.duplicated().sum().item()

34335

In [180]:
df = df.drop_duplicates()

In [181]:
df.isnull().sum()

invoice              0
stock_code           0
description       4275
quantity             0
invoice_date         0
price                0
customer_id     235151
country              0
dtype: int64

In [182]:
df = df.dropna()

In [183]:
df.invoice = df.invoice.astype(str)

In [184]:
n0=len(df.stock_code.unique())
n0

4646

In [185]:
mask = df['stock_code'].fillna('').astype(str).str.count(r'\D') >= 2
result = df.loc[mask, 'stock_code'].unique()
result

array(['POST', '79323LP', '15056BL', '79323GR', 'BANK CHARGES', 'TEST001',
       'TEST002', 'PADS', 'ADJUST', 'ADJUST2', 'SP1002', 'DOT', 'CRUK'],
      dtype=object)

In [186]:
short_codes = df[df['stock_code'].astype(str).str.len() <= 2]['stock_code'].unique()
print(sorted(short_codes))

['C2', 'D', 'M']


In [187]:
service_codes = ['POST',  'BANK CHARGES', 'TEST001','TEST002', 'PADS', 'ADJUST', 'ADJUST2', 'SP1002', 'DOT', 'CRUK', 'C2', 'D', 'M']
df = df[~df['stock_code'].astype(str).isin(service_codes)]

In [188]:
n1=len(df.stock_code.unique())
n1

4633

In [189]:
df.shape

(794223, 8)

In [190]:
df.describe()

,quantity,invoice_date,price,customer_id
count,794223.000000,794223,794223.000000,794223.000000
mean,12.652839,2011-01-02 12:55:30.921391,2.971190,15319.640134
min,-80995.000000,2009-12-01 07:45:00,0.000000,12346.000000
25%,2.000000,2010-07-02 09:47:00,1.250000,13971.000000
50%,5.000000,2010-12-02 12:18:00,1.950000,15244.000000
75%,12.000000,2011-07-31 15:50:00,3.750000,16791.000000
max,80995.000000,2011-12-09 12:50:00,649.500000,18287.000000
std,192.078928,NaN,4.474925,1693.639880


In [191]:
# Total unique invoices
total_invoices = df['invoice'].nunique()

# Total cancelled unique invoices (start with 'C')
cancelled_invoices = df[df['invoice'].str.startswith('C')]['invoice'].nunique()

# Return rate
return_rate = cancelled_invoices / total_invoices

print(f"Total invoices: {total_invoices}")
print(f"Cancelled invoices: {cancelled_invoices}")
print(f"Return rate (by invoices): {return_rate:.2%}")

Total invoices: 43879
Cancelled invoices: 7282
Return rate (by invoices): 16.60%


In [192]:
# Total sold quantity (sales)
total_sold = df[df['quantity'] > 0]['quantity'].sum()

# Total returned quantity (returned)
total_returned = abs(df[df['quantity'] < 0]['quantity'].sum())

return_rate_items = total_returned / total_sold

print(f"Total sold: {total_sold}")
print(f"Total returned: {total_returned}")
print(f"Return rate (by quantity): {return_rate_items:.2%}")

Total sold: 10513468
Total returned: 464292
Return rate (by quantity): 4.42%


In [193]:
df['revenue'] = df['quantity'] * df['price']

sales_revenue = df[df['quantity'] > 0]['revenue'].sum()
returns_revenue = -df[df['quantity'] < 0]['revenue'].sum()
return_rate_revenue = returns_revenue / sales_revenue

print(f"Return rate (by revenue): {return_rate_revenue:.2%}")

Return rate (by revenue): 4.16%


In [194]:
df = df[~df['invoice'].str.startswith('C')]

In [195]:
df = df[df['quantity'] > 0]

In [196]:
df = df[df['price'] > 0]

In [197]:
df['revenue'] = df['quantity'] * df['price']

In [198]:
df['customer_id'] = df['customer_id'].astype(int).astype(str)

In [199]:
df.shape

(776577, 9)

In [200]:
df['customer_id'].nunique()

5852

In [201]:
customer_df = df.groupby('customer_id').agg(
    total_revenue = ('revenue', 'sum'),
    num_invoices = ('invoice', 'nunique'),
    num_items = ('quantity', 'sum')
).reset_index()
customer_df.head()

,customer_id,total_revenue,num_invoices,num_items
0,12346,77352.96,3,74239
1,12347,4921.53,8,2967
2,12348,1658.40,5,2704
3,12349,3678.69,3,1621
4,12350,294.40,1,196


In [202]:
# AOV stands for Average Order Value, which is a metric that indicates the average 
# amount of money each customer spends per transaction. It is commonly used in e-commerce 
# to assess customer purchasing behavior.
customer_df['aov'] = customer_df['total_revenue'] / customer_df['num_invoices']

In [203]:
print(customer_df.head())
print(customer_df.shape)

  customer_id  total_revenue  num_invoices  num_items          aov
0       12346       77352.96             3      74239  25784.32000
1       12347        4921.53             8       2967    615.19125
2       12348        1658.40             5       2704    331.68000
3       12349        3678.69             3       1621   1226.23000
4       12350         294.40             1        196    294.40000
(5852, 5)


In [204]:
np.random.seed(42)  # для воспроизводимости

customer_df['group'] = np.random.choice(['A', 'B'], size=len(customer_df), p=[0.5, 0.5])

print(customer_df['group'].value_counts())

group
A    2933
B    2919
Name: count, dtype: int64


In [205]:
# standard deviation is a numerical measure that describes the amount of variation or dispersion in a set of data values. 
# In simple terms, it tells you how close the individual data points are to the mean (average) of the dataset. 
# A low standard deviation indicates that the data points tend to be close to the mean, 
# while a high standard deviation implies that the data points are spread out over a wide range of values.

print(customer_df.groupby('group')['total_revenue'].agg(['mean', 'median', 'std', 'count']))
print(customer_df.groupby('group')['aov'].agg(['mean', 'median', 'std', 'count']))

              mean  median           std  count
group                                          
A      2743.603900  834.44  10307.037884   2933
B      3090.639853  876.57  17424.674540   2919
             mean    median          std  count
group                                          
A      365.372109  269.7125   701.227320   2933
B      403.529010  284.3200  1631.558223   2919


In [206]:
customer_df['total_revenue'].describe(percentiles=[.5, .9, .95, .99, .999])

count      5852.000000
mean       2916.706762
std       14306.854987
min           2.950000
50%         856.020000
90%        5392.771000
95%        9270.270500
99%       28631.081400
99.9%    175635.535180
max      580987.040000
Name: total_revenue, dtype: float64

In [207]:
customer_df.nlargest(10, 'total_revenue')[['customer_id', 'total_revenue']]

,customer_id,total_revenue
5666,18102,580987.04
2262,14646,526751.52
1778,14156,303069.88
2520,14911,272252.79
5026,17450,244784.25
1321,13694,195640.69
5085,17511,172132.87
4037,16446,168472.50
4271,16684,147142.77
67,12415,144033.37


In [208]:
# Sort clients by revenue in descending order
cust_sorted = customer_df.sort_values('total_revenue', ascending=False).reset_index(drop=True)

# Total revenue
total_revenue = cust_sorted['total_revenue'].sum()

# Cumulative revenue and share
cust_sorted['cum_revenue'] = cust_sorted['total_revenue'].cumsum()
cust_sorted['cum_share'] = cust_sorted['cum_revenue'] / total_revenue

# Find the first customer with a cumulative share >= 50%
half_index = (cust_sorted['cum_share'] >= 0.5).idxmax()
n_half = half_index + 1  # +1 because the index is 0

print(f"Total customers: {len(cust_sorted)}")
print(f"Clients generating 50% of revenue: {n_half}")
print(f"This represents {n_half / len(cust_sorted) * 100:.2f}% of all clients.")

Total customers: 5852
Clients generating 50% of revenue: 258
This represents 4.41% of all clients.


In [209]:
# Create revenue deciles (10 groups of equal size)
customer_df['revenue_decile'] = pd.qcut(
    customer_df['total_revenue'],
    q=10,
    labels=False,
    duplicates='drop'
)

# Let's look at the number of clients in each decile and the average revenue there.
decile_stats = customer_df.groupby('revenue_decile', as_index=False)['total_revenue'].agg(['count', 'mean', 'min', 'max'])
print(decile_stats)

   revenue_decile  count          mean      min        max
0               0    586    106.449795     2.95     161.35
1               1    585    218.126923   161.38     285.30
2               2    585    342.112564   285.56     404.30
3               3    585    496.520291   404.40     603.42
4               4    585    724.322427   603.46     856.01
5               5    585   1023.389897   856.03    1207.24
6               6    585   1477.384598  1207.35    1793.22
7               7    585   2273.290598  1794.32    2885.11
8               8    585   3873.634444  2887.99    5381.08
9               9    586  18609.814113  5394.07  580987.04


In [210]:
np.random.seed(42)
customer_df['group'] = 'A'

for decile in customer_df['revenue_decile'].unique():
    idx = customer_df[customer_df['revenue_decile'] == decile].index
    n_b = len(idx) // 2
    b_idx = np.random.choice(idx, size=n_b, replace=False)
    customer_df.loc[b_idx, 'group'] = 'B'

print(customer_df['group'].value_counts())
print(customer_df.groupby('group')['total_revenue'].agg(['mean', 'median', 'std', 'count']))

group
A    2930
B    2922
Name: count, dtype: int64
              mean   median           std  count
group                                           
A      3052.544280  856.020  15248.429373   2930
B      2780.497341  856.585  13297.157232   2922


In [211]:
group_a = customer_df[customer_df['group'] == 'A']['total_revenue']
group_b = customer_df[customer_df['group'] == 'B']['total_revenue']

# Mann-Whitney U (non-parametric, robust to outliers)
stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')


# The t-test takes into account the difference in means, the sample size, and the data dispersion, and yields a p-value.
# Welch's t-test (parametric, but does not assume equal variances)
stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)

print(f"Mann-Whitney U p-value: {p_mw:.4f}")
print(f"Welch's t-test p-value: {p_t:.4f}")

Mann-Whitney U p-value: 0.9710
Welch's t-test p-value: 0.4670


In [212]:
# Simulating a discount: +5% to Group B revenue
customer_df['revenue_simulated'] = customer_df['total_revenue']

mask_b = customer_df['group'] == 'B'
customer_df.loc[mask_b, 'revenue_simulated'] = customer_df.loc[mask_b, 'total_revenue'] * 1.05

# Checking the mean
mean_a = customer_df[customer_df['group'] == 'A']['revenue_simulated'].mean()
mean_b = customer_df[customer_df['group'] == 'B']['revenue_simulated'].mean()

print(f"Mean A: {mean_a:.2f}")
print(f"Mean B: {mean_b:.2f}")
print(f"Absolute Uplift: {mean_b - mean_a:.2f}")
print(f"Relative Uplift: {(mean_b - mean_a) / mean_a:.2%}")

Mean A: 3052.54
Mean B: 2919.52
Absolute Uplift: -133.02
Relative Uplift: -4.36%


In [213]:
group_a = customer_df[customer_df['group'] == 'A']['revenue_simulated']
group_b = customer_df[customer_df['group'] == 'B']['revenue_simulated']

stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)
stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')

print(f"Welch's t-test p-value: {p_t:.4f}")
print(f"Mann-Whitney U p-value: {p_mw:.4f}")

Welch's t-test p-value: 0.7278
Mann-Whitney U p-value: 0.1892


In [214]:
# Stratification by 50 quantiles
customer_df['revenue_strata'] = pd.qcut(
    customer_df['total_revenue'],
    q=50,
    labels=False,
    duplicates='drop'
)

np.random.seed(42)
customer_df['group'] = 'A'
for strata in customer_df['revenue_strata'].unique():
    idx = customer_df[customer_df['revenue_strata'] == strata].index
    n_b = len(idx) // 2
    b_idx = np.random.choice(idx, size=n_b, replace=False)
    customer_df.loc[b_idx, 'group'] = 'B'

# A/A test
group_a = customer_df[customer_df['group'] == 'A']['total_revenue']
group_b = customer_df[customer_df['group'] == 'B']['total_revenue']

stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')
stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)

print(customer_df.groupby('group', as_index = False)['total_revenue'].agg(['mean', 'median', 'std', 'count']))
print(f"Mann-Whitney p: {p_mw:.4f}")
print(f"Welch's t-test p: {p_t:.4f}")

  group         mean  median           std  count
0     A  3056.131431  856.03  17304.214665   2949
1     B  2775.072814  856.01  10414.661516   2903
Mann-Whitney p: 0.9676
Welch's t-test p: 0.4508


In [215]:
# Looking at the 99th percentile
p99 = customer_df['total_revenue'].quantile(0.99)
print(f"99th percentile: {p99:.2f}")

# Winsorization: we cut off everything above the p99.
customer_df['total_revenue_wins'] = customer_df['total_revenue'].clip(upper=p99)

# Checking the result
print(customer_df['total_revenue_wins'].describe(percentiles=[.5, .9, .99]))

99th percentile: 28631.08
count     5852.000000
mean      2269.493110
std       4173.274556
min          2.950000
50%        856.020000
90%       5392.771000
99%      28507.915686
max      28631.081400
Name: total_revenue_wins, dtype: float64


In [216]:
customer_df['revenue_strata'] = pd.qcut(
    customer_df['total_revenue_wins'],
    q=50,
    labels=False,
    duplicates='drop'
)

np.random.seed(42)
customer_df['group'] = 'A'
for strata in customer_df['revenue_strata'].unique():
    idx = customer_df[customer_df['revenue_strata'] == strata].index
    n_b = len(idx) // 2
    b_idx = np.random.choice(idx, size=n_b, replace=False)
    customer_df.loc[b_idx, 'group'] = 'B'

# A/A-test
group_a = customer_df[customer_df['group'] == 'A']['total_revenue_wins']
group_b = customer_df[customer_df['group'] == 'B']['total_revenue_wins']

stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')
stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)

print(customer_df.groupby('group', as_index=False)['total_revenue_wins'].agg(['mean', 'median', 'std', 'count']))
print(f"Mann-Whitney p: {p_mw:.4f}")
print(f"Welch's t-test p: {p_t:.4f}")

  group         mean  median          std  count
0     A  2267.082160  856.03  4147.867268   2949
1     B  2271.942264  856.01  4199.640332   2903
Mann-Whitney p: 0.9673
Welch's t-test p: 0.9645


In [217]:
customer_df['revenue_simulated'] = customer_df['total_revenue_wins']
mask_b = customer_df['group'] == 'B'
customer_df.loc[mask_b, 'revenue_simulated'] = customer_df.loc[mask_b, 'total_revenue_wins'] * 1.05

mean_a = customer_df[customer_df['group'] == 'A']['revenue_simulated'].mean()
mean_b = customer_df[customer_df['group'] == 'B']['revenue_simulated'].mean()

print(f"Mean A: {mean_a:.2f}")
print(f"Mean B: {mean_b:.2f}")
print(f"Absolute Uplift: {mean_b - mean_a:.2f}")
print(f"Relative Uplift: {(mean_b - mean_a) / mean_a:.2%}")

group_a = customer_df[customer_df['group'] == 'A']['revenue_simulated']
group_b = customer_df[customer_df['group'] == 'B']['revenue_simulated']

stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)
stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')

print(f"Welch's t-test p: {p_t:.4f}")
print(f"Mann-Whitney p: {p_mw:.4f}")

Mean A: 2267.08
Mean B: 2385.54
Absolute Uplift: 118.46
Relative Uplift: 5.23%
Welch's t-test p: 0.2900
Mann-Whitney p: 0.1894


In [218]:
np.random.seed(42)

group_a = customer_df[customer_df['group'] == 'A']['revenue_simulated'].values
group_b = customer_df[customer_df['group'] == 'B']['revenue_simulated'].values

n_bootstrap = 10000
diffs = np.zeros(n_bootstrap)

for i in range(n_bootstrap):
    sample_a = np.random.choice(group_a, size=len(group_a), replace=True)
    sample_b = np.random.choice(group_b, size=len(group_b), replace=True)
    diffs[i] = sample_b.mean() - sample_a.mean()

ci_lower = np.percentile(diffs, 2.5)
ci_upper = np.percentile(diffs, 97.5)

print(f"Mean difference: {diffs.mean():.2f}")
print(f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"Contains 0: {ci_lower <= 0 <= ci_upper}")

Mean difference: 119.31
95% CI: [-102.62, 338.27]
Contains 0: True


In [219]:
# Parameters
effect_size = 0.05  # Relative Uplift 5%
alpha = 0.05
power = 0.8

# Converting the relative effect to Cohen's d
# Cohen's d = (mean_B - mean_A) / pooled_std
mean_a = customer_df[customer_df['group'] == 'A']['revenue_simulated'].mean()
std_a = customer_df[customer_df['group'] == 'A']['revenue_simulated'].std()
std_b = customer_df[customer_df['group'] == 'B']['revenue_simulated'].std()

pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)
cohens_d = (mean_a * 0.05) / pooled_std

print(f"Mean A: {mean_a:.2f}")
print(f"Pooled std: {pooled_std:.2f}")
print(f"Cohen's d: {cohens_d:.4f}")

# Sample Size Calculation
analysis = TTestIndPower()
n_required = analysis.solve_power(
    effect_size=cohens_d,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

print(f"We need customers in each group: {int(np.ceil(n_required))}")
print(f"Total customers: {int(np.ceil(n_required)) * 2}")
print(f"We have: {len(customer_df[customer_df['group'] == 'A'])} + {len(customer_df[customer_df['group'] == 'B'])}")

Mean A: 2267.08
Pooled std: 4280.75
Cohen's d: 0.0265
We need customers in each group: 22389
Total customers: 44778
We have: 2949 + 2903


In [220]:
# Base return rate at the customer level (simulated)
# Assume the customer return rate follows a Beta(2, 30) distribution — mean ~6.25%
np.random.seed(42)

# For Group A — baseline return rate
customer_df['return_rate'] = np.random.beta(2, 30, size=len(customer_df))

# For Group B — increase by 2% (simulating a degradation of the guardrail)
mask_b = customer_df['group'] == 'B'
customer_df.loc[mask_b, 'return_rate'] = customer_df.loc[mask_b, 'return_rate'] * 1.02

# Comparing
mean_a = customer_df[customer_df['group'] == 'A']['return_rate'].mean()
mean_b = customer_df[customer_df['group'] == 'B']['return_rate'].mean()

print(f"Return rate A: {mean_a:.4%}")
print(f"Return rate B: {mean_b:.4%}")
print(f"Absolute Uplift: {(mean_b - mean_a):.4%}")
print(f"Relative Uplift: {(mean_b - mean_a) / mean_a:.2%}")

# t-test
group_a = customer_df[customer_df['group'] == 'A']['return_rate']
group_b = customer_df[customer_df['group'] == 'B']['return_rate']
stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)
print(f"Welch's t-test p: {p_t:.4f}")

Return rate A: 6.1906%
Return rate B: 6.3762%
Absolute Uplift: 0.1856%
Relative Uplift: 3.00%
Welch's t-test p: 0.0868


In [221]:
# Net revenue for each customer
# return_rate already includes the simulated increase for group B
customer_df['net_revenue'] = customer_df['revenue_simulated'] * (1 - customer_df['return_rate'])

# Comparing groups
net_a = customer_df[customer_df['group'] == 'A']['net_revenue'].mean()
net_b = customer_df[customer_df['group'] == 'B']['net_revenue'].mean()

print(f"Net Revenue A: {net_a:.2f}")
print(f"Net Revenue B: {net_b:.2f}")
print(f"Absolute Uplift: {net_b - net_a:.2f}")
print(f"Relative Uplift: {(net_b - net_a) / net_a:.2%}")

# t-test for net revenue
group_a = customer_df[customer_df['group'] == 'A']['net_revenue']
group_b = customer_df[customer_df['group'] == 'B']['net_revenue']

stat_t, p_t = ttest_ind(group_a, group_b, equal_var=False)
stat_mw, p_mw = mannwhitneyu(group_a, group_b, alternative='two-sided')

print(f"Welch's t-test p: {p_t:.4f}")
print(f"Mann-Whitney p: {p_mw:.4f}")

Net Revenue A: 2123.02
Net Revenue B: 2228.54
Absolute Uplift: 105.52
Relative Uplift: 4.97%
Welch's t-test p: 0.3128
Mann-Whitney p: 0.2187


In [222]:
np.random.seed(42)
group_a = customer_df[customer_df['group'] == 'A']['net_revenue'].values
group_b = customer_df[customer_df['group'] == 'B']['net_revenue'].values

n_bootstrap = 10000
diffs = np.zeros(n_bootstrap)

for i in range(n_bootstrap):
    sample_a = np.random.choice(group_a, size=len(group_a), replace=True)
    sample_b = np.random.choice(group_b, size=len(group_b), replace=True)
    diffs[i] = sample_b.mean() - sample_a.mean()

ci_lower = np.percentile(diffs, 2.5)
ci_upper = np.percentile(diffs, 97.5)

print(f"Mean difference in net revenue: {diffs.mean():.2f}")
print(f"95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"Contains 0: {ci_lower <= 0 <= ci_upper}")

Mean difference in net revenue: 106.39
95% CI: [-99.96, 311.55]
Contains 0: True


In [223]:
customer_df.to_parquet('data/processed/customer_df.parquet')